# Setup

### Description

This is the first run of our machine translation model from CS224N Spring 2024. The goal is to implement a sequence-to-sequence model with attention for translating sentences from Chinese to English .

### 01 Install required packages

In [ ]:
# Install required packages
# %pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp
%pip install -q docopt sentencepiece sacrebleu tensorboard
%pip install -q requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.3 MB/s eta 0:00:00


### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
!git clone https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024.git

Cloning into 'CS224N-NLP-with-DL-2024'...
remote: Enumerating objects: 270, done.
remote: Total 270 (delta 0), reused 0 (delta 0), pack-reused 270 (from 2)
Receiving objects: 100% (270/270), 62.12 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (100/100), done.


### 03 Switch to the project directory

In [3]:
import os

# Move into your specific project folder on the remote machine
os.chdir("/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-3")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-3

=== Available Project Files ===
total 66980
drwxr-xr-x 6 root root     4096 Aug 11 08:54 .
drwxr-xr-x 3 root root     4096 Aug 11 08:54 ..
-rw-r--r-- 1 root root    17076 Aug 11 08:54 01_implementation_v1.ipynb
-rw-r--r-- 1 root root     9014 Aug 11 08:54 02_training_local.ipynb
-rw-r--r-- 1 root root     9813 Aug 11 08:54 03_experiments_colab_v1.ipynb
-rw-r--r-- 1 root root  1235360 Aug 11 08:54 a3_spr24_student_handout.pdf
-rw-r--r-- 1 root root     1299 Aug 11 08:54 beam_search_diagnostics.py
-rw-r--r-- 1 root root      111 Aug 11 08:54 collect_submission.bat
-rw-r--r-- 1 root root       99 Aug 11 08:54 collect_submission.sh
-rw-r--r-- 1 root root      124 Aug 11 08:54 env-cpu.yml
-rw-r--r-- 1 root root      158 Aug 11 08:54 env-gpu.yml
-rw-r--r-- 1 root root        0 Aug 11 08:54 __init__.py
-rw-r--r-- 1 root root     2324 Aug 11 08:54 model_embeddings.py
-rw-r--r-- 1 root root    31954 Au

### 04 Import libraries

In [4]:
%load_ext autoreload
%autoreload 2

import torch

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

### 05 Set logging levels [OPTIONAL]

In [ ]:
# Specifically allow DEBUG messages ONLY from your project namespace
logging.getLogger("nmt").setLevel(logging.DEBUG)

In [ ]:
# Specifically disallow DEBUG messages ONLY from your project namespace
logging.getLogger("nmt").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [5]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy

=== CPU Specifications ===
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
NUMA node0 CPU(s):                       0,1

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       996Mi       6.9Gi       2.0Mi       4.8Gi        11Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   47G   66G  42% /


### 07 Verify GPU Availability [OPTIONAL]

In [6]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: Tesla T4
CUDA Capability: (7, 5)


### 08 End the session [OPTIONAL]

In [36]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [21]:
!git pull

remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 3), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 10.31 KiB | 2.58 MiB/s, done.
From https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024
   d2243a0..cb653f7  main       -> origin/main
Updating d2243a0..cb653f7
Fast-forward
 .../assignment-3/03_experiments_colab_v1.ipynb     | 2222 ++++++--------------
 .../2024/assignment-3/download_artifacts.py        |   48 +
 2 files changed, 711 insertions(+), 1559 deletions(-)
 create mode 100644 02-assignments/2024/assignment-3/download_artifacts.py


# 1 The First Run on a Toy Dataset

## 01 Run for a single epoch with a toy dataset

In [14]:
!bash run_toy_colab.sh

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 10:22:33.040784: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
uniformly initialize parameters [-0.100000, +0.100000]
use device: cuda:0
begin Maximum Likelihood training
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
epoch 1, iter 25, avg. loss 7.90, avg. ppl 2709.68 cum. examples 200, speed 2527.46 words/sec, time elapsed 2.08 sec
epoch 1, iter 50, avg. loss 6.88, avg. ppl 973

In [15]:
!python run.py decode \
    toy_model_colab.bin \
    ./zh_en_data/dev.zh \
    ./zh_en_data/dev.en \
    outputs/toy_dev_outputs.txt \
    --cuda

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 10:29:19.690438: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
load test source sentences from [./zh_en_data/dev.zh]
load test target sentences from [./zh_en_data/dev.en]
load model from toy_model_colab.bin
Decoding:   0% 0/1001 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
Decoding: 100% 1001/1001 [00:34<00:00, 28.92it/s]
Corpus BLEU: 0.2961716216061359


## 02 Run on a full dataset for 1 epoch

In [10]:
!mkdir -p outputs

In [11]:
!bash run_epoch_1.sh train

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 09:01:59.905125: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
uniformly initialize parameters [-0.100000, +0.100000]
use device: cuda:0
begin Maximum Likelihood training
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
epoch 1, iter 10, avg. loss 7.95, avg. ppl 2848.57 cum. examples 320, speed 4492.43 words/sec, time elapsed 1.88 sec
epoch 1, iter 20, avg. loss 7.07, avg. ppl 117

## 03 Run on a full dataset for 30 epochs

In [16]:
!bash run.sh train

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 10:31:24.687509: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
uniformly initialize parameters [-0.100000, +0.100000]
use device: cuda:0
begin Maximum Likelihood training
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
epoch 1, iter 10, avg. loss 7.95, avg. ppl 2848.57 cum. examples 320, speed 4454.18 words/sec, time elapsed 1.89 sec
epoch 1, iter 20, avg. loss 7.07, avg. ppl 117

In [17]:
!bash run.sh dev

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 12:00:59.037667: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
load test source sentences from [./zh_en_data/dev.zh]
load test target sentences from [./zh_en_data/dev.en]
load model from model.bin
Decoding:   0% 0/1001 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
Decoding: 100% 1001/1001 [00:38<00:00, 26.10it/s]
Corpus BLEU: 21.84168178780005


In [18]:
!bash run.sh test

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2026-08-11 12:02:01.352136: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
load test source sentences from [./zh_en_data/test.zh]
load test target sentences from [./zh_en_data/test.en]
load model from model.bin
Decoding:   0% 0/1001 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv1d(
Decoding: 100% 1001/1001 [00:39<00:00, 25.60it/s]
Corpus BLEU: 21.00815630029154


In [19]:
from pathlib import Path

for path in [
    "model.bin",
    "model.bin.optim",
    "outputs/dev_outputs.txt",
    "outputs/test_outputs.txt",
]:
    file_path = Path(path)
    print(
        f"{path}: "
        f"{file_path.stat().st_size / 1024**2:.2f} MB"
        if file_path.exists()
        else f"{path}: missing"
    )

model.bin: 237.86 MB
model.bin.optim: 474.13 MB
outputs/dev_outputs.txt: 0.10 MB
outputs/test_outputs.txt: 0.10 MB


In [22]:
!python download_artifacts.py

Created assignment3_artifacts.zip (221.05 MB)
Traceback (most recent call last):
  File "/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-3/download_artifacts.py", line 46, in <module>
    files.download(str(ARCHIVE))
  File "/usr/local/lib/python3.12/dist-packages/google/colab/files.py", line 232, in download
    comm_manager = _IPython.get_ipython().kernel.comm_manager
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'kernel'


In [23]:
from google.colab import files; files.download("assignment3_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!ls

01_implementation_v1.ipynb     run_epoch_1.sh
02_training_local.ipynb        run.py
03_experiments_colab_v1.ipynb  runs
a3_spr24_student_handout.pdf   run.sh
assignment3_artifacts.zip      run_toy_colab.sh
beam_search_diagnostics.py     sanity_check_en_es_data
collect_submission.bat	       sanity_check.py
collect_submission.sh	       src.model
download_artifacts.py	       src.vocab
env-cpu.yml		       tgt.model
env-gpu.yml		       tgt.vocab
__init__.py		       toy_model.bin
model.bin		       toy_model.bin.optim
model.bin.optim		       toy_model_colab.bin
model_embeddings.py	       toy_model_colab.bin.optim
nmt_model.py		       train_toy_mps.sh
outputs			       utils.py
__pycache__		       vocab.json
requirements.txt	       vocab.py
run.bat			       zh_en_data


In [33]:
!curl --fail --upload-file assignment3_artifacts.zip \
  https://transfer.sh/assignment3_artifacts.zip

curl: (7) Failed to connect to transfer.sh port 443 after 178 ms: Connection refused


In [34]:
from getpass import getpass
from pathlib import Path
import requests

token = getpass("GitHub token: ")

repository = "ilyarudyak/CS224N-NLP-with-DL-2024"
tag = "colab-artifacts-2026-08-11"
asset_path = Path("assignment3_artifacts.zip")

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {token}",
    "X-GitHub-Api-Version": "2026-03-10",
}

# Find the release created in the browser.
release_response = requests.get(
    f"https://api.github.com/repos/{repository}/releases/tags/{tag}",
    headers=headers,
)
release_response.raise_for_status()
release = release_response.json()

# GitHub returns an upload URL containing a placeholder such as {?name,label}.
upload_url = release["upload_url"].split("{")[0]

with asset_path.open("rb") as asset_file:
    upload_response = requests.post(
        upload_url,
        params={"name": asset_path.name},
        headers={
            **headers,
            "Content-Type": "application/zip",
        },
        data=asset_file,
    )

upload_response.raise_for_status()
uploaded_asset = upload_response.json()

print("Uploaded successfully.")
print("Download URL:", uploaded_asset["browser_download_url"])

Uploaded successfully.
Download URL: https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024/releases/download/colab-artifacts-2026-08-11/assignment3_artifacts.zip
